# EDA for Telco Customer Churn dataset

[dataset link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

[dataset description](https://community.ibm.com/community/user/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)

In [ ]:
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import numpy as np 

pd.set_option('display.max_columns', 200)
np.random.seed(42)

## Download dataset

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Prelimitary analys

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
is_na_default = df.isna().sum()
is_na_default.name = "is_na_default"

is_na_space = df.astype(str).map(lambda x: x == "" or x == " ").sum()
is_na_space.name = "is_empty_space"

pd.concat([is_na_default, is_na_space], axis = 1)

In [ ]:
df.duplicated().sum()

## Data preparation (basic)

In [ ]:
to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for columnn in to_category_columns:
    df[columnn] = df[columnn].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
    

In [ ]:
df.dtypes

In [ ]:
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.describe()

### Data NA handling

In [ ]:
df["TotalCharges"].isna().sum()

As we see only 11 rows with NA values we can drop them without loosing signifficant insights

In [ ]:
print(df.shape)

df = df.dropna()
df["TotalCharges"].isna().sum()
print(df.shape)

## Data Observation

Check data distribution

In [ ]:
from dataclasses import dataclass
import math

# Utilities for plot generation

@dataclass
class DiscretePlot():
    data: pd.Series
    title: str
    x_title:str | None = None
    y_title: str| None = None
    
def count_plot_sizes(len_plot_map: int) -> tuple[int, int]:
    columns = 2 if len_plot_map == 2 else 3
    if len_plot_map < columns:
        columns = len_plot_map
    
    rows = math.ceil(len_plot_map / columns)
    return rows, columns
    
def plot_discrete_vars(plot_map: list[DiscretePlot]):
    nrows, ncolumns = count_plot_sizes(len(plot_map))
    fig, axs = plt.subplots(nrows, ncolumns, figsize=(5 * ncolumns, 4 * nrows))
    
    for i, plot in enumerate(plot_map):
        ax = axs.flat[i] # type: ignore
        
        counts = plot.data.value_counts()
        sns.barplot(x=counts.index, y=counts.values, ax=ax)
        
        ax.set_title(plot.title)
        ax.set_xlabel(plot.x_title if plot.x_title else "")
        ax.set_ylabel(plot.y_title if plot.y_title else "Count")
        ax.tick_params(axis='x', labelrotation=45)
    
    total_cells = nrows * ncolumns
    if total_cells > len(plot_map):
        for j in range(len(plot_map), total_cells):
            axs.flat[j].set_visible(False) # type: ignore
    
    print("Proceeded", len(plot_map)," maps")
    plt.tight_layout()
    plt.show()

In [ ]:
dicrete_plot_map = [
    DiscretePlot(df["gender"], title="Customer gender"),
    DiscretePlot(df["SeniorCitizen"], title="Customer senior citizen", x_title="Are customers senior citizens"),
    DiscretePlot(df["Partner"], title="Customer with partner", x_title="Do customers have a partner"),
    DiscretePlot(df["Dependents"], title="Customer having dependents", x_title="Do customers have a dependent"),
    DiscretePlot(df["PhoneService"], title="Customer with phone service", x_title="Do customers include phone service plan"),
    DiscretePlot(df["MultipleLines"], title="Customer with multiple lines"),
    DiscretePlot(df["InternetService"], title="Customer internet service type"),
    DiscretePlot(df["OnlineSecurity"], title="Customer with online security", x_title="Do customers buy online security"),
    DiscretePlot(df["OnlineBackup"], title="Customer with online backup", x_title="Do customers buy online beckup"),
    DiscretePlot(df["DeviceProtection"], title="Customer with device protection", x_title="Do customers buy device protection"),
    DiscretePlot(df["TechSupport"], title="Customer with premium tech support", x_title="Do customers buy premium tech support"),
    DiscretePlot(df["StreamingTV"], title="Customer streaming TV", x_title="Do customers stream TV from another providers"),
    DiscretePlot(df["StreamingMovies"], title="Customer streaming movies", x_title="Do customers stream movies from another providers"),
    DiscretePlot(df["Contract"], title="Customer contract types"),
    DiscretePlot(df["PaperlessBilling"], title="Customer use paperless billing"),
    DiscretePlot(df["PaymentMethod"], title="Customer payment method"),
    DiscretePlot(df["Churn"], title="Customer left", x_title="Do customers left this quarter"),
]

plot_discrete_vars(dicrete_plot_map)

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(25,8))
sns.histplot(df.tenure, bins=20, kde=True, ax=axs[0])
axs[0].set_title("Tenure density")


sns.histplot(df["MonthlyCharges"], bins=20, kde=True, ax=axs[1])
axs[1].set_title("Monthly charges density")


sns.histplot(df["TotalCharges"], bins=20, kde=True)
axs[2].set_title("Total charges density")


plt.show()


## Data correlation analys

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

def encode_columns(data, columns_list):
    for column in columns_list:
        data[column] = LabelEncoder().fit_transform(data[column])
    return data

In [ ]:
# Encode binary columns with LabelEncoder
binary_columns =(
        [
                "gender", 
                "SeniorCitizen", 
                "Partner", 
                "Dependents", 
                "PhoneService", 
                "PaperlessBilling", 
                "Churn"
        ])

numeric_columns = (
        [
                "tenure",
                "MonthlyCharges",
                "TotalCharges",
        ]
)

df_enc = df.copy()
df_enc = encode_columns(df_enc, binary_columns)


cols = list(df_enc.select_dtypes('number').columns)
corr = df_enc[cols].corr()

plt.figure(figsize=(10, 8))
ax = sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                 center=0, vmin=-1, vmax=1, square=True)

target = 'Churn'
for i, column in enumerate(ax.get_xticklabels()):
    if column.get_text() == target:
        column.set_color('darkred')
        column.set_fontweight('bold')
        column.set_fontsize(20)
for i, column in enumerate(ax.get_yticklabels()):
    if column.get_text() == target:
        column.set_color('darkred')
        column.set_fontweight('bold')
        column.set_fontsize(20)

plt.title("Correlation heatmap. Only numeric and binary")
plt.show()

See many small correlations with target. And many multicoleniarities (do not cover all category labels yet btw). Sees to better to use stable on multicolearity models

### Nominative feature importance

In [ ]:
from scipy.stats import chi2_contingency # pyright: ignore[reportMissingImports]

# Cramers test
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    phi2 = chi2 / n
    r, k = ct.shape
    return np.sqrt(phi2 / min(k-1, r-1))

# Define nominative features for analys
columns_to_exclude = list(df.select_dtypes('number').columns) + ["customerID", "Churn"]
columns_to_plot = set(df.columns) - set(columns_to_exclude)

def calculate_cramers_on_data(data_frame: pd.DataFrame, columns: set[str]) -> tuple[str, np.float64]:
    """Do a cramer's V test on feature and Churn"""
    cramer_feat_relations = {}
    for col in columns:
        cramer_feat_relations[col] = cramers_v(data_frame[col], data_frame["Churn"])
    return sorted(cramer_feat_relations.items(), key= lambda item: item[1], reverse=True)


In [ ]:
from typing import Literal

def define_relation_color_level(c_relation) ->Literal["green","orange","red"]:
    if c_relation<=0.1:
        return "red"
    if c_relation <= 0.3:
        return "orange"
    else:
        return "green"

def plotPointPlot(df, c_name: str, ax,c_relation: np.float64):
    sns.pointplot(df, x = c_name, y = "Churn", estimator = "mean", ax=ax)
    ax.tick_params(axis='x', labelrotation=45)
    c_rel_level = define_relation_color_level(c_relation)
    ax.set_xlabel("") # Xlabel is excess for plot 
    ax.set_title(f"{c_name} ({c_relation:.2f})", color=c_rel_level)
    return ax

def plotPointPlots(df, plots_with_cramer:tuple[str, np.float64]):
    nrows, ncolumns = count_plot_sizes(len(plots_with_cramer))
    fig, axs = plt.subplots(nrows, ncolumns, figsize=(5 * ncolumns, 4 * nrows))
    
    for i, (c_name, c_relation) in enumerate(plots_with_cramer):
        ax = axs.flat[i] # type: ignore
        plotPointPlot(df, c_name,ax,c_relation)
    
    total_cells = nrows * ncolumns
    if total_cells > len(plots_with_cramer):
        for j in range(len(plots_with_cramer), total_cells):
            axs.flat[j].set_visible(False) # type: ignore
    
    print("Proceeded", len(plots_with_cramer)," plots")
    fig.suptitle("Point plots with Cramer's V relation", y = 1.005)
    plt.tight_layout()
    plt.show()

print("Interpretation: 0.00-0.10 negligible, 0.10-0.30 small, 0.30-0.50 medium, 0.50+ large")
df_encoded_churn = df.copy()
df_encoded_churn["Churn"] = LabelEncoder().fit_transform(df["Churn"])

sorted_data = calculate_cramers_on_data(df, columns_to_plot)
plotPointPlots(df_encoded_churn, sorted_data)


Analyzing the plot we can gather that features from green and yellow groups have good dividing capability. And red ones are giving small effect. Most of the features actually informative and can be used for analys

We see that some features (like electronic check fiber optic and No internet / phone service) gives big impact on prediction. That can lead to less reliable model on low informative features (like model will perform worse with credit card rathen than with electronic check. Sane for no internet customers)

Gender do not impact churn at all (sounds logical). But Parntner, senior and dependents yes.

Actually I think we need an additional analys on customers *with* additional services and(current plot give high weight to people that do not use internet at all). The same for multiple Lines. Check below

In [ ]:
df_cramer_add = df[(df["InternetService"] != "No") & (df["PhoneService"] != "No")]
for col in df_cramer_add.select_dtypes('category').columns:
    df_cramer_add[col] = df_cramer_add[col].cat.remove_unused_categories()

columns_to_plot = columns_to_plot - set(['gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'Contract', 'PaperlessBilling',
    'PaymentMethod'])

sorted_data = calculate_cramers_on_data(df_cramer_add, columns_to_plot)
print("Interpretation: 0.00-0.10 negligible, 0.10-0.30 small, 0.30-0.50 medium, 0.50+ large")
plotPointPlots(df_cramer_add, sorted_data)


Recalculate the cramers's V score for customers that use this services. See that `Streaming movies` and `StreamingTV` score drops to negligible. That mean streaming was not signifficant (but the option of internet was - cramers give big score)

`Multiple lines` score do not signifficatly change. This mean that in our case multiple line option do not affect on churning

In [ ]:
pairplot_fig = sns.pairplot(df.drop("SeniorCitizen", axis=1), kind="kde",hue="Churn", corner=True)
pairplot_fig.fig.suptitle("Data KDE pairplot", y = 1.02)
plt.show()

Densities

Looking at `tenure` density there is a peak of churning from people that are clients less than $\approx10$ month. That means the new one customers are in red zone of churning. Otherwise loyal customers (50+ months with business) tends to continue using service

Looking at `monthlyCharges` density we see correlation: more month charges the more churned customers

Looking at `totalCharges` we see combination of both features (As more `tenure` and `mothlyCharges` as more `totalCharges`). There is a peak of churn rate in 0-1300$ spend in a sum

---

MonthlyChargers vs Tenure (EDA)

This part of plot shows us that customers with high monthly charges and low tenure are frequently churning (look at weights in the left up zone). Monthly charges range at low tenure zone cover the most part of churned customers that is insightful.

Also we see that even loyal customers with big charges have churned (we see some point in the right up corner). 

Interesting that there are almost no one churned in the zone 20+ month and month charges up to 75

Underlining this part. We see that new customer + high charges is a danger zone for churning

---

Tenure vs TotalCharges (EDA)

As $TotalCharges = tenure * monthlyCharges$ (multicollinearity) we see almost linear combination with big weight of customers in zero tenure and zero total charges zone (that proved described pattern of "new customer"). Also we should notice that churned customers spread across high total charges zone.

---

MonthlyCharges vs TotalCharges (EDA)

As $TotalCharges = tenure * monthlyCharges$ we see the coreality from different side (from monthly cahrges side). Biggest weight has zone with high monthly charges and low total charges (and tenure). Also proving the point of "new customer and high bill" theory

In [ ]:
%pip install altair prince vegafusion vl-convert-python -q

In [ ]:
import altair as alt
alt.data_transformers.enable("vegafusion")


In [ ]:
import prince

famd = prince.FAMD(
    n_components=2,
    n_iter=3,
    rescale_with_mean=True,
    rescale_with_std=True,
    copy=True,
    check_input=True,
    random_state=42,
    engine="sklearn", 
    handle_unknown="error",  # same parameter as sklearn.preprocessing.OneHotEncoder
)

X_features = df.drop(columns=['Churn', 'customerID'])

famd = famd.fit(X_features)

df_for_plot = df.set_index('Churn')
famd.plot(df_for_plot, x_component=0, y_component=1, color_rows_by='Churn')

In [ ]:
# Top variables contributing to each component
for i in range(2):
    print(f"Top 5 contributors in component {i} ({famd.percentage_of_variance_[i]:.2f}% of variance)")
    top = famd.column_contributions_.sort_values(i, ascending=False)[i][:5]
    print(top.to_string())
    print()

From the FMAD (Factor analysis of mixed data) we see that dimensial reducing do not accomplish classifing task properly (clusters are mixed up). Component 0 + component 1 give $\approx15$% of churn variance that is not enought to explain target variable

FAMD analys tell us that such columns as `MontlyCharges`, `DeviceProtection`, `TechSupport`, `StreamingMovies`, `StreamingTV`, `OnlineBackup` and `OnlineSecurity`, `InternetService` form an cloud that contain mostly not churned one clients. Must note that most of them are the *optional service* offers and are highly correlated with each other (Without connected internet there will not be others services). FMAD may simply split clients that do not include additional services (and have low monthcharge). Thats why we need to be careful with that FMAD analys.

# Conclusion

## About prelimitary analys

NA's were deleted (only 11 na values in total charges)
everything ok in data pipeline (no duplicates, no misleadings in data) 

## Churning factors

- Tenure. New customers tend to suspend work with a business
- MontlyCharges. Seeing an tendency in churning customes with highel month charges
- Internet factor. Connected internet brings changes in customer churn behavior

### Not signifficant factors

- Gender. Has small effect on spliiting (cramers score) and cant be statistically proved (cover zero by 95% interval)
- StreamingTV / StreamingMovies / MultipleLines. Do not impact on customer churn if the phone internet / phone service included. But the internet connection itself is an great factor (that why this StreamingX factor was calculated as signifficant firstly)

## Multicollinearity

Data have extreme amount of multicollineariry (see heatmap). Should keep notion while model will be developing

## FMAD

- Dimension reduction explain only 15% of variance (do not split classes properly)
- Additional services form a cluster mostly not churned clients

## Recomendations

- Multicollinearity stable models (trees, ensembles) are prefered
- Gender, PhoneCervice and MultipleLines can be excluded at all
- Total charge is excessive due to linear interpretaiion of monthcharge and tenure